In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
train_df = pd.read_csv('../data/train_clean.csv')
val_df = pd.read_csv('../data/val_clean.csv')
test_df = pd.read_csv('../data/test_clean.csv')

train_df.head()


,text,label
0,Bernie Sanders candidacy is attractive to many...,0
1,RIYADH (Reuters) - Saudi King Salman received ...,1
2,"Right before eighth grade, Trump s father sent...",0
3,Someone we haven t heard a lot from during the...,0
4,"BRIDGEWATER, N.J. (Reuters) - President Donald...",1


In [3]:
X_train = train_df['text']
y_train = train_df['label']

X_val = val_df['text']
y_val = val_df['label']

X_test = test_df['text']
y_test = test_df['label']


In [4]:
tfidf = TfidfVectorizer(stop_words='english', max_features=50000)
X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)
X_test_vec = tfidf.transform(X_test)


ValueError: np.nan is an invalid document, expected byte or unicode string.

In [5]:
import pandas as pd

train_df = pd.read_csv('../data/train_clean.csv')
val_df = pd.read_csv('../data/val_clean.csv')
test_df = pd.read_csv('../data/test_clean.csv')

# Make sure text is string and not NaN
for split_df in (train_df, val_df, test_df):
    split_df['text'] = split_df['text'].fillna('').astype(str)
    # Drop completely empty texts
    split_df.drop(split_df[split_df['text'].str.strip() == ''].index, inplace=True)

# Prepare X, y
X_train = train_df['text']
y_train = train_df['label']

X_val = val_df['text']
y_val = val_df['label']

X_test = test_df['text']
y_test = test_df['label']

print(len(X_train), len(X_val), len(X_test))


35178 4396 4400


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=50000,
    min_df=2  # ignore extremely rare tokens to avoid empty vocab issues
)

X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)
X_test_vec = tfidf.transform(X_test)

X_train_vec.shape, X_val_vec.shape, X_test_vec.shape


((35178, 50000), (4396, 50000), (4400, 50000))

In [7]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train_vec, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [8]:
y_pred = model.predict(X_val_vec)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

accuracy, precision, recall, f1


(0.9820291173794359,
 0.9788930581613509,
 0.9839698255539839,
 0.9814248765577239)

In [9]:
import joblib

joblib.dump(model, '../backend/app/baseline_model.pkl')
joblib.dump(tfidf, '../backend/app/tfidf_vectorizer.pkl')


['../backend/app/tfidf_vectorizer.pkl']